# 012 · First-Order Linear Differential Equations

| | |
|---|---|
| **Track** | Python for Calculus |
| **Level** | 4 — Transforms and differential equations |
| **Prerequisites** | 007 and 009. Lesson 011 explains why a transform can turn this into algebra. This lesson solves the equations directly. |
| **Next** | This is the last lesson of the current foundation. |

**Place in the path.** The standard form is

\[
\frac{dy}{dx} + P(x)\, y = Q(x)
\]

Every solution in this lesson is checked twice. SymPy's `dsolve` produces a symbolic solution and the initial condition is substituted back into the differential equation.
SciPy's `solve_ivp` produces a numerical solution. The maximum gap between the two, on a grid, has to be small.

## Learning objectives

By the end of this lesson you can:

- put a first-order equation into standard form
- solve \(y' = y\), \(y' + 2y = 4\), and \(y' + y = x\) with initial conditions
- explain the integrating factor on the constant-coefficient example
- compare a symbolic solution with a numerical one on the same axes
- see what changes when the initial condition is withheld: a free constant remains


## 1. The simplest growth equation

**See.**

\[
\frac{dy}{dx} = y, \qquad y(0) = 1
\]

In standard form, \(y' - y = 0\), so \(P(x) = -1\) and \(Q(x) = 0\).
The solution is \(y = e^{x}\). At \(x = 0\) the exponential is 1, which matches the initial condition.

**Name.** `sp.Function("y")` creates an unknown function, not a symbol.
`sp.dsolve(equation, y(x), ics={y(0): 1})` solves the initial-value problem.
`ics` is what kills the constant of integration. Without it, you get \(y = C e^{x}\).

**Check.** Substitute \(e^{x}\) back into \(y' - y\) and get 0. Then compare with `solve_ivp` on \([0, 3]\).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp

x = sp.symbols("x")
y = sp.Function("y")

ode = sp.Eq(y(x).diff(x) - y(x), 0)
general = sp.dsolve(ode, y(x))
particular = sp.dsolve(ode, y(x), ics={y(0): 1})
print("without an initial condition:", general)
print("with y(0) = 1:", particular)

assert particular.rhs.equals(sp.exp(x))
residual = sp.diff(particular.rhs, x) - particular.rhs
assert sp.simplify(residual) == 0
assert particular.rhs.subs(x, 0) == 1
constants = general.rhs.free_symbols - {x}
assert len(constants) == 1

xs = np.linspace(0, 3, 200)
numeric = solve_ivp(lambda t, state: state, (0, 3), [1.0], t_eval=xs, rtol=1e-8, atol=1e-8)
gap = float(np.max(np.abs(numeric.y[0] - np.exp(xs))))
print(f"max |numeric - exp(x)| = {gap:.3e}")
assert numeric.success
assert gap < 1e-5

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, np.exp(xs), color="#1f4e79", linewidth=2, label=r"symbolic $e^{x}$")
ax.plot(xs, numeric.y[0], color="#b85c38", linestyle="--", linewidth=2, label="solve_ivp")
ax.set_title(r"$y' = y,\ y(0)=1$. Two solvers, one solution.")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


**Read the output.** The general solution still contains a constant. The initial condition sets that constant to 1.
The dashed numerical curve sits on the solid exponential. The printed gap is the second method, quantified.


## 2. Constant coefficients, a nonzero right-hand side

**See.**

\[
\frac{dy}{dx} + 2y = 4, \qquad y(0) = 0
\]

**The integrating factor, by hand.** Multiply by \(e^{\int 2\, dx} = e^{2x}\):

\[
\frac{d}{dx}\left(y e^{2x}\right) = 4 e^{2x}
\]

Integrate both sides:

\[
y e^{2x} = 2 e^{2x} + C, \qquad y = 2 + C e^{-2x}
\]

\(y(0) = 0\) forces \(2 + C = 0\), so \(C = -2\) and

\[
y = 2\left(1 - e^{-2x}\right)
\]

The equilibrium is \(y = 2\): the solution starts at 0 and approaches 2. The picture should show that approach, not a line.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp

x = sp.symbols("x")
y = sp.Function("y")

ode = sp.Eq(y(x).diff(x) + 2 * y(x), 4)
particular = sp.dsolve(ode, y(x), ics={y(0): 0})
expected = 2 * (1 - sp.exp(-2 * x))
print(particular)
assert sp.simplify(particular.rhs - expected) == 0

residual = sp.diff(expected, x) + 2 * expected - 4
assert sp.simplify(residual) == 0
assert expected.subs(x, 0) == 0

xs = np.linspace(0, 4, 250)
numeric = solve_ivp(lambda t, state: 4 - 2 * state, (0, 4), [0.0], t_eval=xs, rtol=1e-8, atol=1e-8)
exact = 2 * (1 - np.exp(-2 * xs))
gap = float(np.max(np.abs(numeric.y[0] - exact)))
print(f"max gap = {gap:.3e}")
assert numeric.success and gap < 1e-5

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, exact, color="#1f4e79", linewidth=2, label=r"$2(1-e^{-2x})$")
ax.plot(xs, numeric.y[0], color="#b85c38", linestyle="--", linewidth=2, label="solve_ivp")
ax.axhline(2, color="gray", linestyle=":", label="equilibrium y = 2")
ax.set_title(r"$y' + 2y = 4,\ y(0)=0$ approaches the equilibrium")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 3. A varying right-hand side

**See.**

\[
\frac{dy}{dx} + y = x, \qquad y(0) = 0
\]

Here \(P(x) = 1\) is still constant, and \(Q(x) = x\) is not.
The integrating factor is \(e^{x}\):

\[
\frac{d}{dx}\left(y e^{x}\right) = x e^{x}
\]

The integral on the right is the integration by parts from lesson 008: \(\int x e^{x} dx = e^{x}(x-1)\).

\[
y e^{x} = e^{x}(x - 1) + C, \qquad y = x - 1 + C e^{-x}
\]

\(y(0) = 0\) forces \(C = 1\), so \(y = x - 1 + e^{-x}\).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp

x = sp.symbols("x")
y = sp.Function("y")

ode = sp.Eq(y(x).diff(x) + y(x), x)
particular = sp.dsolve(ode, y(x), ics={y(0): 0})
expected = x - 1 + sp.exp(-x)
print(particular)
assert sp.simplify(particular.rhs - expected) == 0
residual = sp.diff(expected, x) + expected - x
assert sp.simplify(residual) == 0
assert sp.simplify(expected.subs(x, 0)) == 0

xs = np.linspace(0, 5, 250)
numeric = solve_ivp(lambda t, state: t - state, (0, 5), [0.0], t_eval=xs, rtol=1e-8, atol=1e-8)
exact = xs - 1 + np.exp(-xs)
gap = float(np.max(np.abs(numeric.y[0] - exact)))
print(f"max gap = {gap:.3e}")
assert numeric.success and gap < 1e-5

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, exact, color="#1f4e79", linewidth=2, label=r"$x - 1 + e^{-x}$")
ax.plot(xs, numeric.y[0], color="#b85c38", linestyle="--", linewidth=2, label="solve_ivp")
ax.set_title(r"$y' + y = x,\ y(0)=0$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print("three initial-value problems agree symbolically and numerically")


### Own it

Solve \(y' + y = 0\) with \(y(0) = 3\).
The integrating factor is again \(e^{x}\), and the solution is \(y = 3e^{-x}\).
Check the residual and the initial condition. A numerical comparison is optional here; the residual is the exact check.


In [ ]:
import sympy as sp

x = sp.symbols("x")
y = sp.Function("y")
ode = sp.Eq(y(x).diff(x) + y(x), 0)
particular = sp.dsolve(ode, y(x), ics={y(0): 3})
expected = 3 * sp.exp(-x)
print(particular)
assert sp.simplify(particular.rhs - expected) == 0
assert sp.simplify(sp.diff(expected, x) + expected) == 0
assert expected.subs(x, 0) == 3
print("checkpoint passed")


## Mistakes this lesson is meant to prevent

- Forgetting to move every term that involves \(y\) to the left side before naming \(P(x)\).
- Dropping the initial condition and then treating a one-parameter family as the solution of an initial-value problem.
- Trusting `dsolve` without substituting the result back into the differential equation.
- Plotting only the numerical curve. The curve needs the symbolic formula in the legend, or the picture does not say what it claims to be.
- Comparing the solvers by eye alone. Print the maximum gap.

## What you now have

You can solve a first-order linear initial-value problem by an integrating factor, confirm it with SymPy,
and audit it with a numerical solver. That is the standard this foundation uses everywhere:
a definition or a hand method, a library call, and a second representation that has to agree.

The twelve lessons are the base of the library. Later courses — series, multivariable calculus, numerical methods —
should keep this contract: one idea at a time, a worked example, a check by a second method, and a stated place in the path.
